# Atividade 07.2 — Classificação KNN
## Dataset: Dry Bean Dataset

O dataset possui **13.611 amostras**, **16 features morfológicas** e **7 classes** de feijão:
`SEKER`, `BARBUNYA`, `BOMBAY`, `CALI`, `HOROZ`, `SIRA`, `DERMASON`.


## 0 - Instalação e Importações

In [86]:
!pip install -q scikit-learn openpyxl

In [87]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

## 1 - Carregamento e Exploração do Dataset

In [88]:
# Carregar o dataset a partir do arquivo Excel
df = pd.read_excel("Dry_Bean_Dataset.xlsx")

print(f"Shape do dataset: {df.shape}")
print(f"\nColunas: {df.columns.tolist()}")
print(f"\nClasses encontradas: {sorted(df['Class'].unique())}")
print(f"\nDistribuição das classes:")
print(df['Class'].value_counts())

Shape do dataset: (13611, 17)

Colunas: ['Area', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength', 'AspectRation', 'Eccentricity', 'ConvexArea', 'EquivDiameter', 'Extent', 'Solidity', 'roundness', 'Compactness', 'ShapeFactor1', 'ShapeFactor2', 'ShapeFactor3', 'ShapeFactor4', 'Class']

Classes encontradas: ['BARBUNYA', 'BOMBAY', 'CALI', 'DERMASON', 'HOROZ', 'SEKER', 'SIRA']

Distribuição das classes:
Class
DERMASON    3546
SIRA        2636
SEKER       2027
HOROZ       1928
CALI        1630
BARBUNYA    1322
BOMBAY       522
Name: count, dtype: int64


In [89]:
# Verificar valores ausentes
missing = df.isnull().sum()
threshold = 0.5 * len(df)
cols_to_drop = missing[missing > threshold].index.tolist()

print(f"Colunas com >50% ausentes (removidas): {cols_to_drop if cols_to_drop else 'Nenhuma'}")

df_clean = df.drop(columns=cols_to_drop)
print(f"Shape após limpeza: {df_clean.shape}")
print(f"Valores ausentes totais: {df_clean.isnull().sum().sum()}")

Colunas com >50% ausentes (removidas): Nenhuma
Shape após limpeza: (13611, 17)
Valores ausentes totais: 0


## 2 - Separação de Features e Target (com LabelEncoder)


In [90]:
# Separar features e target
X = df_clean.drop(columns=['Class'])
y_raw = df_clean['Class']

# Codificar classes textuais -> inteiros
le = LabelEncoder()
y = le.fit_transform(y_raw)

print("Mapeamento de classes (Label → Inteiro):")
for i, cls in enumerate(le.classes_):
    print(f"  {cls:>10} -> {i}")

print(f"\nTotal de features: {X.shape[1]}")
print(f"Total de amostras: {len(y)}")

Mapeamento de classes (Label → Inteiro):
    BARBUNYA -> 0
      BOMBAY -> 1
        CALI -> 2
    DERMASON -> 3
       HOROZ -> 4
       SEKER -> 5
        SIRA -> 6

Total de features: 16
Total de amostras: 13611


## 3 - Divisão Treino / Validação / Teste (Estratificada)

In [91]:
# 70% treino, 15% validação, 15% teste (estratificado por classe)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# Normalização Min-Max — APENAS no treino
train_min = X_train.min()
train_range = (X_train.max() - train_min).replace(0, 1)

X_train = (X_train - train_min) / train_range
X_val   = (X_val   - train_min) / train_range
X_test  = (X_test  - train_min) / train_range

n = len(y)
print(f"Treino:    {len(X_train):>5} amostras ({len(X_train)/n:.0%})")
print(f"Validação: {len(X_val):>5} amostras ({len(X_val)/n:.0%})")
print(f"Teste:     {len(X_test):>5} amostras ({len(X_test)/n:.0%})")

Treino:     9527 amostras (70%)
Validação:  2042 amostras (15%)
Teste:      2042 amostras (15%)


## 4 - Implementação do KNN


In [92]:
class KNN:
    """K-Nearest Neighbors"""

    def __init__(self, k=3):
        self.k = k

    def fit(self, X, y):
        self.X_train = np.array(X)
        self.y_train = np.array(y)

    def predict(self, X):
        X = np.array(X)
        return np.array([self._predict(x) for x in X])

    def _predict(self, x):
        # Distância Euclidiana para todos os pontos de treino
        distances = np.sqrt(np.sum((self.X_train - x) ** 2, axis=1))
        # Índices dos K vizinhos mais próximos
        k_indices = np.argsort(distances)[:self.k]
        k_labels  = self.y_train[k_indices]
        # Votação: retorna a classe com mais votos
        classes, counts = np.unique(k_labels, return_counts=True)
        return classes[np.argmax(counts)]

def accuracy_score(y_true, y_pred):
    y_true = np.array(y_true).ravel()
    y_pred = np.array(y_pred).ravel()
    return np.mean(y_true == y_pred)

## 5 - Busca pelo Melhor K (no conjunto de Validação)

> O dataset possui 13.611 amostras. Por eficiência, testamos apenas **K ímpares de 1 a 31**.  

In [94]:
# Testamos K ímpares de 1 a 31
k_values = [k for k in range(1, 32) if k % 2 != 0]
val_scores = []

print(f"Testando K = {k_values}\n")
print(f"{'K':>4}  {'Acurácia Validação':>20}")
print("-" * 30)

for k in k_values:
    model = KNN(k=k)
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    acc   = accuracy_score(y_val, preds)
    val_scores.append(acc)
    print(f"{k:>4}  {acc:>20.4f}")

best_k  = k_values[np.argmax(val_scores)]
best_acc = max(val_scores)
print(f"\nMelhor K: {best_k}  |  Acurácia na validação: {best_acc:.4f}")

Testando K = [1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29, 31]

   K    Acurácia Validação
------------------------------
   1                0.8991
   3                0.9104
   5                0.9109
   7                0.9119
   9                0.9153
  11                0.9177
  13                0.9177
  15                0.9197
  17                0.9187
  19                0.9197
  21                0.9212
  23                0.9202
  25                0.9187
  27                0.9182
  29                0.9182
  31                0.9187

Melhor K: 21  |  Acurácia na validação: 0.9212


## 6 - Avaliação Final no Conjunto de Teste

In [95]:
# Treino final: TREINO + VALIDAÇÃO após escolha do melhor K
X_train_final = pd.concat([X_train, X_val], axis=0)
y_train_final = np.concatenate([y_train, y_val])

final_model = KNN(k=best_k)
final_model.fit(X_train_final, y_train_final)

# Predição no TESTE
y_pred = final_model.predict(X_test)

final_acc = accuracy_score(y_test, y_pred)
print(f"\n{'='*45}")
print(f"   RESULTADO FINAL — Conjunto de Teste")
print(f"{'='*45}")
print(f"   Melhor K:        {best_k}")
print(f"   Acurácia Final:  {final_acc:.4f} ({final_acc*100:.2f}%)")
print(f"{'='*45}")


   RESULTADO FINAL — Conjunto de Teste
   Melhor K:        21
   Acurácia Final:  0.9158 (91.58%)
